# W7｜NumPy Debug：偏差最早发生在哪一步？

程序能跑完，数值却不对。用固定输入复现，在数组发生变化的地方停住，检查 `shape`、中间数组和计算轴，再修复并换一份输入验证。

## 本周路线

1. **筛选与广播**：有效的 `0` 被掩码排除；按列校正却沿错方向广播。
2. **轴与索引**：滑动窗口沿错轴求和；排序后为每台设备选中了旧记录。
3. **迭代与统计**：回溯试点走错起点；组内均值与整体均值有两处独立故障。


## 如何使用

1. 在 VS Code 选择 `week-07/.venv` 的 Python kernel。每题先读需求，运行输入格，再正常运行 Playground，记下实际输出。起始断言失败是练习材料。
2. 在 Playground 左侧点一个断点，从运行按钮旁的菜单选择 **Debug Cell**。暂停后在 **Variables** 看数组及其 `shape`；用 **Step Over** 走过一行、**Step Into** 进入函数、**Call Stack** 查看调用者，最后用 **Continue** 继续。
3. 在 Playground 修改并重跑。把修好的实现填到「函数组装」，再运行「单元检查」。组装格起初只有函数壳；参考解答折叠在每题末尾。

**同桌检查**：轮流展示固定输入、停点、当时看到的数组或变量、改动，以及同一输入重跑的输出。还没修完，就展示已经查过的一步，并说出下一步要看的值。


## 开始前：只用到这些 NumPy 记号

每题输入格都会运行 `import numpy as np`。输入均为 NumPy 数组，不需要处理非法类型或无穷大，也不要原地改动输入。

```python
x = np.array([0.0, np.nan, 6.0])
x.shape                     # (3,)：一维，3 个元素
valid = ~np.isnan(x)         # [True, False, True]：布尔掩码
x[valid]                    # [0., 6.]：按掩码选出有效观测

matrix = np.array([[1., 2.], [3., 4.], [5., 6.]])
matrix.shape                # (3, 2)：3 行、2 列
matrix.sum(axis=0)          # [9., 12.]：沿行方向累加，剩下每列的和
matrix.sum(axis=1)          # [3., 7., 11.]：沿列方向累加，剩下每行的和
```

`np.nan` 表示缺失值；判断缺失用 `np.isnan(x)`，不能用 `x == np.nan`。比较浮点数组时用 `np.testing.assert_allclose(actual, expected, equal_nan=True)`；它会在不相符时报错。遇到陌生数组，先看 `shape`，再看一两行内容。


## 场景一｜有效的 `0` 怎么消失了？

### 补充知识｜为什么要小心 NaN

`NaN` 是 **Not a Number** 的缩写，是 IEEE 754 标准中的特殊浮点值。它既可能表示缺失观测，也可能由无效的实数运算产生。例如 `np.sqrt(-1.)` 在默认设置下会发出警告并返回 `nan`，程序仍可能继续运行。

- **比较规则与普通数不同。** NumPy 中 `NaN` 连自己也不相等，用 `== np.nan` 找不到它，用 `!= np.nan` 也筛不掉它。
- **数值异常会继续传播。** 含 `NaN` 的数组直接求均值会得到 `NaN`，加法等许多后续运算也会继续得到 `NaN`。
- **真假判断识别不了缺失。** Python 中 `bool(np.nan)` 是 `True`，而有效读数 `0` 的真假值却是 `False`。

```python
import numpy as np

x = np.array([0., np.nan, 6.])
x == np.nan        # [False, False, False]
x != np.nan        # [True, True, True]，缺失值也留下了
np.isnan(x)        # [False, True, False]，专门判断 NaN
np.mean(x)         # nan
bool(np.nan)       # True
```

R 中也需要专门的判断函数，但比较的显示结果与 Python 不同。

```r
NaN == NaN             # NA，比较结果也是未知的
is.nan(NaN)            # TRUE
mean(c(0, NaN, 6))     # NaN
```

本题明确把原始缺测记为 `np.nan`，允许排除这些位置。若原始数据正常、计算中途才出现 `NaN`，应先查它从哪一步产生，再决定如何处理；直接填成 `0` 或忽略，会改变统计对象。

参考 [NumPy 的 NaN 定义](https://numpy.org/doc/stable/reference/constants.html#numpy.nan)、[缺失值检测](https://numpy.org/doc/stable/reference/generated/numpy.isnan.html)与 [R 的比较规则](https://stat.ethz.ch/R-manual/R-devel/library/base/html/Comparison.html)。

### 场景与需求

一维数组保存仪器读数，`np.nan` 表示缺失，`0` 是有效观测。均值只使用有效观测；没有有效观测时返回 `np.nan`。

### 输入与输出

- `np.array([0., np.nan, 6.])` → `3.0`。
- `[0., 0.]` → `0.0`；`[np.nan, np.nan]` 或空数组 → `np.nan`。

### 你的任务

实现 `mean_observed(values)`：输入为一维浮点数组，输出为一个浮点数，不改动输入。先区分「有效观测在筛选时丢了」和「观测都在，均值算错了」。在掩码生成后停住，检查 `valid` 与 `observed`。


In [ ]:
import numpy as np

q1_values = np.array([0.0, np.nan, 6.0])
q1_expected = 3.0
print("读数", q1_values, "预期均值", q1_expected)


In [ ]:
def mean_observed_trial(values):
    valid = (~np.isnan(values)) & (values != 0)
    observed = values[valid]
    if observed.size == 0:
        return np.nan
    return float(observed.mean())


q1_actual = mean_observed_trial(q1_values)
print("实际均值", q1_actual)
np.testing.assert_allclose(q1_actual, q1_expected, equal_nan=True)


In [ ]:
def mean_observed(values):
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q1_cases = [
    (np.array([0.0, np.nan, 6.0]), 3.0),
    (np.array([0.0, 0.0]), 0.0),
    (np.array([np.nan, np.nan]), np.nan),
    (np.array([], dtype=float), np.nan),
    (np.array([-4.0, np.nan, 2.0]), -1.0),
]
for values, expected in q1_cases:
    before = values.copy()
    actual = mean_observed(values)
    assert isinstance(actual, (float, np.floating)), "应返回一个浮点数"
    np.testing.assert_allclose(actual, expected, equal_nan=True)
    np.testing.assert_allclose(values, before, equal_nan=True)
print("场景一：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在 `observed = values[valid]` 左侧设断点。这一行还没执行，先在 **Variables** 看 `valid`：对 `[0., nan, 6.]`，预期 `[True, False, True]`，实际 `[False, False, True]`。
2. 点 **Step Over**（网页版 **Next**）走过筛选，`observed` 只有 `[6.]`，预期是 `[0., 6.]`。第一次偏差在掩码，求均值之前零已经丢了。

看完清除断点并继续到本格结束，原版断言失败是预期。按证据修好 Playground 后重跑，再把实现写入「函数组装」，运行「单元检查」核对原例和变式。

```python
def mean_observed(values):
    valid = ~np.isnan(values)
    observed = values[valid]
    if observed.size == 0:
        return np.nan
    return float(observed.mean())
```

#### 错误原因

`valid` 实际为 `[False, False, True]`，所以 `observed` 只剩 `[6.]`。条件 `values != 0` 把有效的零也排除了。

#### 修正前后的本质差别

掩码只回答「是否缺失」，不按数值是否为零筛选。均值公式没有变，送进去的观测改为 `[0., 6.]`。没有观测时显式返回 `np.nan`。

</details>


## 场景二｜每列一个偏移，怎么校正到了每行？

### 补充知识｜行、列和多维数组

#### 二维数组怎样表示行与列

NumPy 数组的类型是 `ndarray`。用嵌套列表创建二维数组时，**内层每个列表是一行**。`shape` 按轴的顺序记录长度，二维时就是 `(行数, 列数)`；索引从 `0` 开始，`a[i, j]` 取第 `i` 行、第 `j` 列。

```python
import numpy as np

a = np.array([[0, 1, 2],
              [3, 4, 5]])
a.shape       # (2, 3)：2 行、3 列
a.ndim        # 2：共有两个轴
a[1, 2]       # 5
a[0, :]       # [0, 1, 2]：取一行
a[:, 0]       # [0, 3]：取一列
```

- `axis=0` 是行下标所在的轴，`axis=1` 是列下标所在的轴。
- **沿哪个轴求和，就合并哪个轴**。`a.sum(axis=0)` 合并两行，留下每列的和 `[3, 5, 7]`；`a.sum(axis=1)` 合并三列，留下每行的和 `[3, 12]`。
- 行与列表示什么，由数据的使用者约定。本题规定「行是观测，列是传感器」。

#### 一维数组需要显式增加行列方向

```python
v = np.array([0, 1, 2])
v.shape                # (3,)
v.ndim                 # 1：只有 axis 0，尚无行列之分
v[None, :].shape        # (1, 3)：一行
v[:, None].shape        # (3, 1)：一列
```

数学中三个分量的向量属于三维向量空间；这里 `v.ndim` 仍然是 `1`。数组的轴数和向量的分量数是两个概念。索引里的 `None` 会插入一个长度为 `1` 的轴，已有元素的值保持原样。

#### reshape 的排列与转置

`reshape` 默认用 `order="C"` 的索引顺序，**最后一个轴的下标变化最快**。二维时可以理解为先沿一行排列，再进入下一行。元素总数必须与新形状的乘积一致。

```python
np.arange(6).reshape(2, 3)  # [[0, 1, 2], [3, 4, 5]]
a.reshape(3, 2)            # [[0, 1], [2, 3], [4, 5]]
a.T                        # [[0, 3], [1, 4], [2, 5]]
```

后两项都是 `(3, 2)`，元素对应的位置却不同。想交换行列时用转置。`order="F"` 则让第一个轴的下标变化最快，二维时对应先沿列排列。这里说的是索引与元素的排列顺序，实际内存布局还要结合数组视图来看。

#### 三维和更高维怎样初始化

三维数组可以看成若干张二维表叠起来，`shape` 继续按轴列出长度。数值计算、机器学习里常把这样的多维数组称为**张量**，本课操作的 NumPy 对象统一是 `ndarray`。

```python
t = np.arange(24).reshape(2, 3, 4)
t.shape       # (2, 3, 4)：2 块，每块 3 行、4 列
t[0]          # [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
t[1]          # [[12, 13, 14, 15], [16, 17, 18, 19], [20, 21, 22, 23]]
t[1, 0, 2]   # 14：第 1 块、第 0 行、第 2 列

np.zeros((2, 3, 4))        # 同样形状，全部初始化为 0
np.ones((2, 3, 4))         # 全部初始化为 1
np.full((2, 3, 4), np.nan) # 全部初始化为缺失值
```

例如把 `(2, 3, 4)` 约定为「两批、每批三次观测、每次四个传感器」，`t[批次, 观测, 传感器]` 就能表示一个具体读数。

参考 [NumPy 数组基础](https://numpy.org/doc/stable/user/absolute_beginners.html)、[reshape 的索引顺序](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html)与 [按形状初始化](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html)。

#### 向量四则运算与常用函数

下面均使用实数 `ndarray`。`+`、`-`、`*`、`/` 在形状相容时逐元素运算，形状不同会按广播规则对齐。

| 符号 | 对应函数 | 含义 |
| --- | --- | --- |
| `u + v` | `np.add(u, v)` | 对应元素相加 |
| `u - v` | `np.subtract(u, v)` | 对应元素相减 |
| `u * v` | `np.multiply(u, v)` | 对应元素相乘 |
| `u / v` | `np.divide(u, v)` | 对应元素相除 |
| `u ** 2` | `np.power(u, 2)` | 每个元素分别平方 |
| `u @ v` | `np.matmul(u, v)` | 按向量或矩阵乘法规则计算 |

```python
u = np.array([1., 2.])
v = np.array([3., 4.])
u + v       # [4., 6.]
u - v       # [-2., -2.]
u * v       # [3., 8.]：逐元素乘
u / v       # [1/3, 1/2]：逐元素除
2 * u       # [2., 4.]：每个元素乘 2
u ** 2      # [1., 4.]
u @ v       # 11.0：两个一维向量的点积
np.dot(u, v)  # 11.0：在这个一维例子中也是点积
np.outer(u, v)  # [[3., 4.], [6., 8.]]：外积
```

`np.sum(u)` 求和，`np.mean(u)` 求均值。`np.abs`、`np.sqrt`、`np.exp`、`np.log` 分别对每个元素取绝对值、平方根、指数和自然对数。`np.linalg.norm(u)` 在这个一维例子中计算欧几里得长度。

#### @ 怎样读取行列关系

- 两个一维数组 `(k,) @ (k,)` 得到一个点积标量。
- 矩阵乘向量 `(m, k) @ (k,)` 得到 `(m,)`，每行与向量各做一次点积。
- 向量乘矩阵 `(k,) @ (k, n)` 得到 `(n,)`，向量与每列各做一次点积。
- 两个矩阵 `(m, k) @ (k, n)` 得到 `(m, n)`，中间的长度 `k` 必须相等。

```python
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[2., 0.], [1., 2.]])
A * B             # [[2., 0.], [3., 8.]]
A @ B             # [[4., 4.], [10., 8.]]
A @ u             # [5., 11.]
u @ A             # [7., 10.]
np.matmul(A, B)   # 与 A @ B 相同

b = np.array([5., 11.])
np.linalg.solve(A, b)  # [1., 2.]：A 为可逆方阵，求解 A @ x = b
```

这里 `A / b` 仍是逐元素除法并应用广播。线性方程的求解使用 `np.linalg.solve`。`np.dot` 在更高维时有自己的轴规则，不能把它与 `@` 当成所有情况下都相同的写法。

两边均至少二维时，`@` 将最后两个轴当作矩阵轴，前面的轴按广播规则配对。例如 `(2, 3, 4) @ (4, 5)` 得到 `(2, 3, 5)`，可以看成两份矩阵共用一组变换。

参考 [NumPy 的矩阵乘法](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html)、[dot 的轴规则](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)与 [线性方程求解](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html)。

### 场景与需求

矩阵 `raw` 的**行是观测，列是传感器**，形状为 `(n, p)`。一维数组 `offset` 的形状为 `(p,)`，保存每列传感器的固定偏移。每个元素的校正值为 `raw[i, j] - offset[j]`。

### 输入与输出

```python
raw = np.array([[12., 20.], [14., 24.]])
offset = np.array([2., 10.])
# 预期：[[10., 10.], [12., 14.]]，形状仍为 (2, 2)
```

### 你的任务

实现 `calibrate_channels(raw, offset)`，返回同形状的浮点矩阵，不改动两个输入。继续调用辅助函数 `correct_reading(raw, offset)`。它直接接收两个原数组时检查通过，包装调用却失败：在辅助函数中停住两次，查看 `offset.shape`，并从调用栈找到形状在哪一层发生了变化。


In [ ]:
import numpy as np

q2_raw = np.array([[12.0, 20.0], [14.0, 24.0]])
q2_offset = np.array([2.0, 10.0])
q2_expected = np.array([[10.0, 10.0], [12.0, 14.0]])
print("原始矩阵\n", q2_raw, "\n各列偏移", q2_offset)
print("预期校正\n", q2_expected)


In [ ]:
def correct_reading_trial(raw, offset):
    return raw - offset


def calibrate_channels_trial(raw, offset):
    column_offset = offset[:, None]
    corrected = correct_reading_trial(raw, column_offset)
    return corrected


np.testing.assert_allclose(
    correct_reading_trial(q2_raw, q2_offset), q2_expected, equal_nan=True
)
print("直接调用校正函数通过")
q2_actual = calibrate_channels_trial(q2_raw, q2_offset)
print("实际校正\n", q2_actual)
np.testing.assert_allclose(q2_actual, q2_expected, equal_nan=True)


In [ ]:
def correct_reading(raw, offset):
    return raw - offset


def calibrate_channels(raw, offset):
    # 调用上面的 correct_reading。
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q2_cases = [
    (q2_raw, q2_offset, q2_expected),
    (
        np.array([[12.0, 20.0], [14.0, 24.0], [16.0, 30.0]]),
        np.array([2.0, 10.0]),
        np.array([[10.0, 10.0], [12.0, 14.0], [14.0, 20.0]]),
    ),
    (np.array([[4.0], [0.0], [-2.0]]), np.array([-2.0]),
     np.array([[6.0], [2.0], [0.0]])),
    (np.empty((0, 2)), np.array([2.0, 10.0]), np.empty((0, 2))),
]
for raw, offset, expected in q2_cases:
    raw_before, offset_before = raw.copy(), offset.copy()
    np.testing.assert_allclose(correct_reading(raw, offset), expected, equal_nan=True)
    actual = calibrate_channels(raw, offset)
    assert isinstance(actual, np.ndarray) and actual.shape == raw.shape
    np.testing.assert_allclose(actual, expected, equal_nan=True)
    np.testing.assert_allclose(raw, raw_before, equal_nan=True)
    np.testing.assert_allclose(offset, offset_before, equal_nan=True)
print("场景二：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在辅助函数的 `return raw - offset` 左侧设断点。第一次从 Playground 直接调用，`offset` 为 `[2., 10.]`，形状 `(2,)`，正好每列一个偏移。
2. 点 **Continue** 到同一行的第二次暂停：`offset` 变成 `[[2.], [10.]]`，形状 `(2, 1)`。在 **Call Stack** 选上一层 `calibrate_channels_trial`，看到高亮的 `corrected = correct_reading_trial(raw, column_offset)`，再向上找 `offset[:, None]`。
3. 同一条减法，直接调用得到正确的 `[[10., 10.], [12., 14.]]`，包装调用却得到 `[[10., 18.], [4., 14.]]`。证据指向调用处改变了偏移方向。

看完清除断点并继续到本格结束，原版断言失败是预期。按证据修好 Playground 后重跑，再把实现写入「函数组装」，运行「单元检查」核对原例和变式。

```python
def correct_reading(raw, offset):
    return raw - offset


def calibrate_channels(raw, offset):
    corrected = correct_reading(raw, offset)
    return corrected
```

#### 错误原因

直接调用时 `offset.shape` 为 `(2,)`，两个偏移分别对应两列。包装函数把它改成 `(2, 1)` 后，减法广播成了每行减同一个偏移。方阵形状碰巧兼容，所以没有报错；换成三行两列，错误方向还会造成形状不兼容。

#### 修正前后的本质差别

传入 `(p,)` 偏移，NumPy 从末尾的列维度匹配，再扩展到所有观测行。辅助函数里的减法保持不变；修正在调用处。

</details>


## 场景三｜窗口都有了，为什么返回的数量不对？

### 补充知识｜把连续窗口排成矩阵

`sliding_window_view` 把连续片段排成窗口数组。对长度为 `n` 的一维数组，当 `1 <= width <= n` 时，结果形状是 `(n - width + 1, width)`：每行是一个窗口，按窗口起点排序，列表示窗口内的位置。窗口每次前移一个元素，相邻行会重叠。下面长度为 `5`、宽度为 `3`，共有 `3` 行；`windows[1]` 是从下标 `1` 开始的窗口。结果与原数组共享数据，默认是只读视图。

```python
import numpy as np
x = np.array([3, 6, 9, 12, 15])
width = 3
windows = np.lib.stride_tricks.sliding_window_view(x, width)
windows.shape     # (3, 3)
windows.tolist()  # [[3, 6, 9], [6, 9, 12], [9, 12, 15]]
windows[1]        # array([6, 9, 12])
```

参考 [NumPy 的滑动窗口视图](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html)。

### 场景与需求

一维浮点数组中，每个长度为 `width` 的完整连续窗口都要计算总和，按窗口起点顺序返回一维浮点数组。`width` 是正整数；超过数组长度时返回空数组。

### 输入与输出

- `[2., 7., 4., 1.]`、宽度 `2` → `[9., 11., 5.]`。
- 同一数组、宽度 `4` → `[14.]`；宽度 `1` → 原来的四个数；宽度 `5` → 空数组。

### 你的任务

实现 `window_sums(values, width)`，不改动输入。`sliding_window_view` 已经生成全部窗口，每行是一个窗口。暂停后检查 `windows.shape` 与 `totals.shape`：需要把哪个维度相加，才会为每行留下一个数？


In [ ]:
import numpy as np

q3_values = np.array([2.0, 7.0, 4.0, 1.0])
q3_width = 2
q3_expected = np.array([9.0, 11.0, 5.0])
print("读数", q3_values, "宽度", q3_width, "预期", q3_expected)


In [ ]:
def window_sums_trial(values, width):
    if width > values.size:
        return np.array([], dtype=float)
    windows = np.lib.stride_tricks.sliding_window_view(values, width)
    totals = windows.sum(axis=0)
    return totals


q3_actual = window_sums_trial(q3_values, q3_width)
print("实际窗口和", q3_actual)
np.testing.assert_allclose(q3_actual, q3_expected, equal_nan=True)


In [ ]:
def window_sums(values, width):
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q3_cases = [
    (q3_values, 2, np.array([9.0, 11.0, 5.0])),
    (q3_values, 4, np.array([14.0])),
    (q3_values, 1, q3_values.copy()),
    (q3_values, 5, np.array([], dtype=float)),
    (np.array([], dtype=float), 1, np.array([], dtype=float)),
    (np.array([5.0, -1.0, 1.0]), 2, np.array([4.0, 0.0])),
]
for values, width, expected in q3_cases:
    before = values.copy()
    actual = window_sums(values, width)
    assert isinstance(actual, np.ndarray) and actual.shape == expected.shape
    assert actual.dtype.kind == "f", "应返回浮点数组"
    np.testing.assert_allclose(actual, expected, equal_nan=True)
    np.testing.assert_allclose(values, before, equal_nan=True)
print("场景三：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在 `totals = windows.sum(axis=0)` 左侧设断点。看 `windows` 是 `[[2., 7.], [7., 4.], [4., 1.]]`，形状 `(3, 2)`，三行各是一个窗口，窗口本身正确。
2. 点 **Step Over**（网页版 **Next**），看 `totals` 实际是 `[13., 12.]`，形状 `(2,)`；预期 `[9., 11., 5.]`，形状 `(3,)`。指出哪条轴被合并了，再决定改哪一行。

看完清除断点并继续到本格结束，原版断言失败是预期。按证据修好 Playground 后重跑，再把实现写入「函数组装」，运行「单元检查」核对原例和变式。

```python
def window_sums(values, width):
    if width > values.size:
        return np.array([], dtype=float)
    windows = np.lib.stride_tricks.sliding_window_view(values, width)
    totals = windows.sum(axis=1)
    return totals
```

#### 错误原因

`windows` 是 `[[2., 7.], [7., 4.], [4., 1.]]`，形状为 `(3, 2)`。沿 `axis=0` 求和得到每列之和 `[13., 12.]`，三个窗口被混在一起了。

#### 修正前后的本质差别

沿 `axis=1` 加总每行的两个元素，留下三个窗口和。窗口已经全部生成；修复的是聚合方向。先处理 `width > n`，也让空输入按约定返回空浮点数组。

</details>


## 场景四｜排序完成，为什么选中了旧记录？

### 补充知识｜排序索引与首次出现的位置

`np.argsort` 返回的是排序所需的下标，不是排好序的值。把同一组下标用于多条数组，原本同位置的数据就会一起移动；下面分数与姓名仍一一对应。一维数组的 `[::-1]` 可把当前顺序反过来。

`np.unique(..., return_index=True)` 默认按值排序去重，并返回各值在当前输入里首次出现的下标。下标按返回值的顺序排列，未必递增：下面 `A` 在下标 `1` 首次出现，`B` 却在下标 `0` 首次出现。

```python
import numpy as np
points = np.array([40, 10, 30])
names = np.array(["林", "陈", "王"])
order = np.argsort(points)  # array([1, 2, 0])
points[order]              # array([10, 30, 40])
names[order]               # array(['陈', '王', '林'])
labels = np.array(["B", "A", "B", "C", "A"])
values, first = np.unique(labels, return_index=True)
values                    # array(['A', 'B', 'C'])
first                     # array([1, 0, 3])
```

参考 [NumPy 的排序索引](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)与 [去重及返回下标](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)。

### 场景与需求

三条同长度的一维数组 `devices`、`times`、`readings`，相同下标表示同一条记录。记录到达顺序可能混乱。每台设备只保留时间最大的记录，同一设备的时间互不相同。

### 输入与输出

```python
devices  = np.array(["A", "B", "A", "B"])
times    = np.array([2, 1, 1, 3])
readings = np.array([20., 7., 10., 9.])
# 预期：{"devices": array(["A", "B"]),
#        "times": array([2, 3]), "readings": array([20., 9.])}
```

### 你的任务

实现 `latest_by_id(devices, times, readings)`，返回上述含三条一维数组的字典，**输出按设备编号字母顺序排列**，不改动输入。`np.argsort` 给出排序索引；`np.unique(..., return_index=True)` 返回排好序的设备名及其首次出现的位置。检查排序后每台设备第一次出现的是新记录还是旧记录，也检查同一排序索引是否同时作用在三条数组上。


In [ ]:
import numpy as np

q4_devices = np.array(["A", "B", "A", "B"])
q4_times = np.array([2, 1, 1, 3])
q4_readings = np.array([20.0, 7.0, 10.0, 9.0])
q4_expected = {
    "devices": np.array(["A", "B"]),
    "times": np.array([2, 3]),
    "readings": np.array([20.0, 9.0]),
}
print("设备", q4_devices, "时刻", q4_times, "读数", q4_readings)
print("预期", q4_expected)


In [ ]:
def latest_by_id_trial(devices, times, readings):
    order = np.argsort(times)
    sorted_devices = devices[order]
    sorted_times = times[order]
    sorted_readings = readings[order]
    unique_devices, first = np.unique(sorted_devices, return_index=True)
    return {
        "devices": unique_devices,
        "times": sorted_times[first],
        "readings": sorted_readings[first],
    }


q4_actual = latest_by_id_trial(q4_devices, q4_times, q4_readings)
print("实际保留的记录", q4_actual)
np.testing.assert_array_equal(q4_actual["devices"], q4_expected["devices"])
np.testing.assert_allclose(q4_actual["times"], q4_expected["times"], equal_nan=True)
np.testing.assert_allclose(q4_actual["readings"], q4_expected["readings"], equal_nan=True)


In [ ]:
def latest_by_id(devices, times, readings):
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q4_cases = [
    (q4_devices, q4_times, q4_readings, q4_expected),
    (q4_devices[::-1], q4_times[::-1], q4_readings[::-1], q4_expected),
    (q4_devices, q4_times.astype(np.uint64), q4_readings, q4_expected),
    (
        np.array(["Z", "A", "Z"]), np.array([4, -1, 2]),
        np.array([-5.0, 8.0, 100.0]),
        {"devices": np.array(["A", "Z"]), "times": np.array([-1, 4]),
         "readings": np.array([8.0, -5.0])},
    ),
    (
        np.array([], dtype=str), np.array([], dtype=int), np.array([], dtype=float),
        {"devices": np.array([], dtype=str), "times": np.array([], dtype=int),
         "readings": np.array([], dtype=float)},
    ),
]
for devices, times, readings, expected in q4_cases:
    before = (devices.copy(), times.copy(), readings.copy())
    actual = latest_by_id(devices, times, readings)
    assert isinstance(actual, dict) and set(actual) == {"devices", "times", "readings"}
    for key in expected:
        assert isinstance(actual[key], np.ndarray)
        assert actual[key].shape == expected[key].shape
    np.testing.assert_array_equal(actual["devices"], expected["devices"])
    np.testing.assert_allclose(actual["times"], expected["times"], equal_nan=True)
    np.testing.assert_allclose(actual["readings"], expected["readings"], equal_nan=True)
    np.testing.assert_array_equal(devices, before[0])
    np.testing.assert_allclose(times, before[1], equal_nan=True)
    np.testing.assert_allclose(readings, before[2], equal_nan=True)
print("场景四：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在 `unique_devices, first = np.unique(sorted_devices, return_index=True)` 左侧设断点。看 `order` 和 `sorted_times`：时刻依次为 `[1, 1, 2, 3]`，旧记录在前。用同一索引核对 `sorted_devices` 与 `sorted_readings`，设备、时刻、读数仍然配对。
2. 点 **Step Over**（网页版 **Next**），看 `first` 在 `sorted_times` 选中了什么。A、B 实际都选到时刻 `1`（读数 `[10., 7.]`），预期时刻 `[2, 3]`（读数 `[20., 9.]`）。设备名没错；偏差来自按升序排列后取首次出现。

看完清除断点并继续到本格结束，原版断言失败是预期。按证据修好 Playground 后重跑，再把实现写入「函数组装」，运行「单元检查」核对原例和变式。

```python
def latest_by_id(devices, times, readings):
    order = np.argsort(times)[::-1]
    sorted_devices = devices[order]
    sorted_times = times[order]
    sorted_readings = readings[order]
    unique_devices, first = np.unique(sorted_devices, return_index=True)
    return {
        "devices": unique_devices,
        "times": sorted_times[first],
        "readings": sorted_readings[first],
    }
```

#### 错误原因

原来的升序排序把旧记录放在前面；`first` 因而选到 A 和 B 在时刻 `1` 的记录。`np.unique` 选的是首次出现的位置，不会替我们比较时间大小。

#### 修正前后的本质差别

时间改为降序排列，每台设备第一次出现的就是最新记录。三条数组始终使用同一个 `order`，保证设备、时刻、读数仍配对；`np.unique` 返回的设备名按字母排序，另外两条输出按对应的 `first` 取值。

</details>


## 场景五｜步长已经减半，试点怎么越走越远？

### 补充知识｜向量长度、平方距离和试点

这里的二维位置写成形状 `(2,)` 的一维数组，两个分量分别是两个坐标。对于实数向量 `d`，`d @ d` 等于各分量平方之和，也等于欧几里得长度的平方。

```python
import numpy as np
x = np.array([0., 3.])
target = np.array([3., -1.])
d = x - target           # [-3., 4.]：两个坐标分别相减
d @ d                    # 25.0
np.sum(d ** 2)           # 25.0
np.linalg.norm(d)        # 5.0

direction = np.array([2., -4.])
x + 0.5 * direction     # [1., 1.]：方向先缩放，再加到位置上
```

多次试点可以先放在 Python 列表里，最后用 `np.array(trials)` 排成二维数组，每行一个试点。若位置函数值已经变小就接受，否则缩小步长再试；这段判断保留顺序循环。

参考 [NumPy 的向量范数](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)。

### 场景与需求

二维位置 `x` 的目标函数为 `f(x) = np.sum((x - [3., -1.]) ** 2)`。给定起点与方向，每次从**同一个起点**计算 `trial = start + step * direction`。步长从 `2` 开始；只有 `f(trial) < f(start)` 才接受，否则减半，最多尝试四次，首次成功就停止。

### 输入与输出

- `start=[0., 0.]`、`direction=[6., -2.]` → 试点 `[[12., -4.], [6., -2.], [3., -1.]]`，第三次接受 `[3., -1.]`。第二次函数值与起点相等，仍应拒绝。
- `start=[4., 0.]`、`direction=[-2., -2.]` → 试点 `[[0., -4.], [2., -2.], [3., -1.]]`，也在第三次接受。

### 你的任务

实现 `try_descent(start, direction)`。两个输入都是形状 `(2,)` 的浮点数组，不改动它们。返回字典：`trials` 是形状 `(k, 2)` 的浮点数组，记录实际尝试的点；`accepted` 是接受的 `(2,)` 数组，四次都失败时为 `None`。这里保留循环，因为下一次是否尝试取决于当前结果。重点在第二次试步暂停，检查步长与试点的出发位置。


In [ ]:
import numpy as np

q5_start = np.array([0.0, 0.0])
q5_direction = np.array([6.0, -2.0])
q5_expected = {
    "trials": np.array([[12.0, -4.0], [6.0, -2.0], [3.0, -1.0]]),
    "accepted": np.array([3.0, -1.0]),
}
print("起点", q5_start, "方向", q5_direction, "预期", q5_expected)


In [ ]:
def try_descent_trial(start, direction):
    target = np.array([3.0, -1.0])
    initial_value = np.sum((start - target) ** 2)
    trial = start.copy()
    step = 2.0
    trials = []
    for attempt in range(1, 5):
        trial = trial + step * direction
        trials.append(trial)
        trial_value = np.sum((trial - target) ** 2)
        if trial_value < initial_value:
            return {"trials": np.array(trials), "accepted": trial}
        step /= 2
    return {"trials": np.array(trials), "accepted": None}


q5_actual = try_descent_trial(q5_start, q5_direction)
print("实际轨迹", q5_actual)
np.testing.assert_allclose(q5_actual["trials"], q5_expected["trials"], equal_nan=True)
np.testing.assert_allclose(q5_actual["accepted"], q5_expected["accepted"], equal_nan=True)


In [ ]:
def try_descent(start, direction):
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q5_cases = [
    (q5_start, q5_direction, q5_expected["trials"], q5_expected["accepted"]),
    (np.array([4.0, 0.0]), np.array([-2.0, -2.0]),
     np.array([[0.0, -4.0], [2.0, -2.0], [3.0, -1.0]]), np.array([3.0, -1.0])),
    (np.array([0.0, 0.0]), np.array([1.0, 0.0]),
     np.array([[2.0, 0.0]]), np.array([2.0, 0.0])),
    (np.array([3.0, -1.0]), np.array([2.0, 0.0]),
     np.array([[7.0, -1.0], [5.0, -1.0], [4.0, -1.0], [3.5, -1.0]]), None),
    (np.array([0.0, 0.0]), np.array([0.0, 0.0]), np.zeros((4, 2)), None),
]
for start, direction, expected_trials, expected_accepted in q5_cases:
    start_before, direction_before = start.copy(), direction.copy()
    actual = try_descent(start, direction)
    assert isinstance(actual, dict) and set(actual) == {"trials", "accepted"}
    assert isinstance(actual["trials"], np.ndarray)
    assert actual["trials"].shape == expected_trials.shape
    np.testing.assert_allclose(actual["trials"], expected_trials, equal_nan=True)
    if expected_accepted is None:
        assert actual["accepted"] is None
    else:
        assert isinstance(actual["accepted"], np.ndarray)
        assert actual["accepted"].shape == (2,)
        np.testing.assert_allclose(actual["accepted"], expected_accepted, equal_nan=True)
    np.testing.assert_allclose(start, start_before, equal_nan=True)
    np.testing.assert_allclose(direction, direction_before, equal_nan=True)
print("场景五：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在 `trial = trial + step * direction` 左侧设断点。第一次停住，`attempt=1`、`step=2`；点 **Step Over**（网页版 **Next**），看到第一点 `[12., -4.]`。
2. 点 **Continue** 到第二轮，`attempt=2`、`step=1`，步长确实减半了。此时 `start` 仍是 `[0., 0.]`，赋值右边读到的旧 `trial` 却是 `[12., -4.]`。
3. 再单步一次，`trial` 实际变成 `[18., -6.]`，从固定起点计算的预期是 `[6., -2.]`。要改的是每轮试步的出发点。

看完清除断点并继续到本格结束，原版断言失败是预期。按证据修好 Playground 后重跑，再把实现写入「函数组装」，运行「单元检查」核对原例和变式。

```python
def try_descent(start, direction):
    target = np.array([3.0, -1.0])
    initial_value = np.sum((start - target) ** 2)
    step = 2.0
    trials = []
    for attempt in range(1, 5):
        trial = start + step * direction
        trials.append(trial)
        trial_value = np.sum((trial - target) ** 2)
        if trial_value < initial_value:
            return {"trials": np.array(trials), "accepted": trial}
        step /= 2
    return {"trials": np.array(trials), "accepted": None}
```

#### 错误原因

第二轮 `step` 已从 `2` 变成 `1`，但出发点仍是上一轮被拒绝的 `[12., -4.]`，于是走到了 `[18., -6.]`。步长更新正确，试点的基准错了。

#### 修正前后的本质差别

每次用固定 `start` 生成一个新数组，只有步长随尝试变化。被拒绝的点留在轨迹中，不参与下一次试点计算，也不原地改变起点。接受条件仍是严格降低函数值。

</details>


## 场景六｜组内均值修好了，整体均值怎么还是错的？

### 补充知识｜掩码计数、按条件取值与有条件除法

布尔数组求和时，`True` 按 `1` 计、`False` 按 `0` 计，可以用来统计选中的元素数。`np.where(条件, 值一, 值二)` 在每个位置按条件选值，返回一个新数组。

```python
import numpy as np
selected = np.array([True, False, True])
selected.sum()                        # 2
np.where(selected, [2., 7., 9.], 0.)  # [2., 0., 9.]
```

当某组没有有效观测时，它的计数为 `0`。先用 `np.full` 建立浮点输出，再让 `np.divide` 只在条件成立的位置计算并写入；条件为假的位置保留输出数组原来的值。

```python
totals = np.array([12., 0., 9.])
counts = np.array([3, 0, 1])
means = np.full(3, np.nan)
np.divide(totals, counts, out=means, where=counts > 0)
means  # [4., nan, 9.]，中间的位置保留为缺失
```

`np.where(counts > 0, totals / counts, np.nan)` 中的除法会先计算，依然可能出现除以零。这里用 `np.divide` 的 `where` 参数控制实际计算的位置。

参考 [NumPy 的按条件选值](https://numpy.org/doc/stable/reference/generated/numpy.where.html)与 [有条件除法](https://numpy.org/doc/stable/reference/generated/numpy.divide.html)。

### 场景与需求

二维浮点矩阵的**每行是一组**，组长不同时用 `np.nan` 补齐。`0` 是有效观测。报告返回 `{"by_group": 一维组均值数组, "overall": 整体均值}`；空组均值为 `np.nan`，所有组都没有有效观测时整体也为 `np.nan`。不改动输入。

### 输入与输出

```python
values = np.array([[0., np.nan, 6.], [12., np.nan, np.nan]])
# 预期：{"by_group": array([3., 12.]), "overall": 6.0}
```

整体使用全部有效元素 `0`、`6`、`12`，每个观测占相同权重。

### 你的任务

实现 `summarize_groups(values)`。这里有两处独立的问题：先用「单组含零」检查掩码，再用「不含零、组长不同」检查整体均值。每修一处就重跑两份输入，记录哪项通过、哪项仍失败。在 `valid`、`counts`、`by_group` 与 `overall` 生成后分别停住，比较它们的含义。


In [ ]:
import numpy as np

q6_cases = [
    (
        "单组含零：只检查筛选",
        np.array([[0.0, np.nan, 6.0]]),
        {"by_group": np.array([3.0]), "overall": 3.0},
    ),
    (
        "不含零、组长不同：只检查整体均值",
        np.array([[2.0, 4.0, 6.0], [12.0, np.nan, np.nan]]),
        {"by_group": np.array([4.0, 12.0]), "overall": 6.0},
    ),
    (
        "同时检查两处",
        np.array([[0.0, np.nan, 6.0], [12.0, np.nan, np.nan]]),
        {"by_group": np.array([3.0, 12.0]), "overall": 6.0},
    ),
]
for label, values, expected in q6_cases:
    print(label, "\n", values, "\n预期", expected)


In [ ]:
def summarize_groups_trial(values):
    valid = (~np.isnan(values)) & (values != 0)
    counts = valid.sum(axis=1)
    totals = np.where(valid, values, 0.0).sum(axis=1)
    by_group = np.full(values.shape[0], np.nan)
    np.divide(totals, counts, out=by_group, where=counts > 0)
    observed_means = by_group[~np.isnan(by_group)]
    overall = float(observed_means.mean()) if observed_means.size else np.nan
    return {"by_group": by_group, "overall": overall}


q6_failed = []
for label, values, expected in q6_cases:
    actual = summarize_groups_trial(values)
    try:
        np.testing.assert_allclose(actual["by_group"], expected["by_group"], equal_nan=True)
        np.testing.assert_allclose(actual["overall"], expected["overall"], equal_nan=True)
        print(label, "通过", "实际", actual)
    except AssertionError:
        print(label, "失败", "实际", actual)
        q6_failed.append(label)
assert not q6_failed, f"未通过的用例：{q6_failed}"


In [ ]:
def summarize_groups(values):
    raise NotImplementedError("填入调试后的实现")


In [ ]:
q6_edge_cases = [
    (np.array([[np.nan, np.nan], [0.0, 4.0], [np.nan, np.nan]]),
     {"by_group": np.array([np.nan, 2.0, np.nan]), "overall": 2.0}),
    (np.full((2, 3), np.nan),
     {"by_group": np.array([np.nan, np.nan]), "overall": np.nan}),
    (np.empty((2, 0)),
     {"by_group": np.array([np.nan, np.nan]), "overall": np.nan}),
    (np.empty((0, 0)),
     {"by_group": np.array([], dtype=float), "overall": np.nan}),
]
q6_all_cases = [(values, expected) for _, values, expected in q6_cases] + q6_edge_cases
for values, expected in q6_all_cases:
    before = values.copy()
    actual = summarize_groups(values)
    assert isinstance(actual, dict) and set(actual) == {"by_group", "overall"}
    assert isinstance(actual["by_group"], np.ndarray)
    assert actual["by_group"].shape == (values.shape[0],)
    assert isinstance(actual["overall"], (float, np.floating))
    np.testing.assert_allclose(actual["by_group"], expected["by_group"], equal_nan=True)
    np.testing.assert_allclose(actual["overall"], expected["overall"], equal_nan=True)
    np.testing.assert_allclose(values, before, equal_nan=True)
print("场景六：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

#### 动手调试

先运行本题「输入」格，对原 Playground 按下面的位置打断点。VS Code 选 **Debug Cell**；浏览器 Jupyter 先启用小虫，若提示重启则确认，补跑「输入」后打断点，再点 **▶** 运行 Playground。断点停在该行执行前。

1. 在 `overall = float(observed_means.mean()) if observed_means.size else np.nan` 左侧设断点。第一次停在「单组含零」：`valid` 是 `[[False, False, True]]`，`counts=[1]`，`by_group=[6.]`；预期应计入零，得到 `counts=[2]`、`by_group=[3.]`。这份输入先暴露筛选错误。
2. 点 **Continue** 到「不含零、组长不同」：`counts=[3, 1]`、`totals=[12., 12.]`、`by_group=[4., 12.]` 都正确。点 **Step Over**（网页版 **Next**），`overall` 却是 `8.0`；按四个观测计算应为 `6.0`。这份输入单独暴露汇总权重错误。
3. 清除断点并继续到结束。先只修掩码重跑：第一项通过，第二项仍失败；再修整体汇总，重跑全部用例。

把验证后的实现写入「函数组装」，再运行「单元检查」，核对同时含两处故障的用例和边界。

```python
def summarize_groups(values):
    valid = ~np.isnan(values)
    counts = valid.sum(axis=1)
    totals = np.where(valid, values, 0.0).sum(axis=1)
    by_group = np.full(values.shape[0], np.nan)
    np.divide(totals, counts, out=by_group, where=counts > 0)
    total_count = counts.sum()
    overall = float(totals.sum() / total_count) if total_count > 0 else np.nan
    return {"by_group": by_group, "overall": overall}
```

#### 错误原因

「单组含零」筛选后只剩 `[6.]`；「不含零、组长不同」的组均值已正确为 `[4., 12.]`，直接平均却得到 `8`。后一份输入实际有四个观测，不应给两组各一半权重。

#### 修正前后的本质差别

掩码只排除 `np.nan`；整体均值改成「全部有效元素的总和 / 全部有效元素的个数」。`np.divide(..., where=counts > 0)` 只计算非空组，让空组保留预先填好的 `np.nan`。只修掩码，第一项通过、第二项仍失败；只修整体汇总则相反，两处都修好才全部通过。

</details>


### 场景六 · 选做｜构造能区分两种算法的矩阵

下面两种算法都接收「每行一组、`np.nan` 补齐」的矩阵；本题限定每行至少有一个有效元素。给定示例中，两种算法都得到 `6`。

换一份你自己的矩阵，让两份代码得到不同结果。先算出预期整体均值，再用断点检查两种算法各自拿了哪些数来求平均。可以改变各行的有效元素数量。

自造输入中每组至少保留一项有效观测，允许用 `np.nan` 补齐不同长度的组。


In [ ]:
def overall_from_groups(values):
    group_means = np.nanmean(values, axis=1)
    return float(group_means.mean())


def overall_from_observations(values):
    observed = values[~np.isnan(values)]
    return float(observed.mean())


# 这份输入两种算法结果相同。替换为自己的矩阵（允许 np.nan 补齐）。
q6_extension_values = np.array([[2.0, 4.0], [8.0, 10.0]])
print("先算组均值", overall_from_groups(q6_extension_values))
print("直接用所有观测", overall_from_observations(q6_extension_values))
